# Experiment 6: Text Processing Models: N-grams, TF-IDF, Word Embeddings and BERT
**Mini-project:** Resume Analyzer Using Natural Language Processing (Group 28)

**Objective:** Build and compare text representation models – an N-gram language model, TF-IDF, Word2Vec, GloVe and contextual BERT embeddings – on our resume dataset.

Open in Colab: [Exp6_Text_Representation_Models.ipynb](https://colab.research.google.com/github/GnsShubham06/resume-analyzer-nlp/blob/main/notebooks/experiments/Exp6_Text_Representation_Models.ipynb) &nbsp;|&nbsp; Run with **Runtime → Run all**.
Data: 100 resumes (20 per domain); the screenshots in the report use five fictional (synthetic) resumes.

In [ ]:
# ---- Setup: runs in Google Colab and locally --------------------------------
# In Colab this cell downloads the project repository (code + anonymised
# dataset) from GitHub and installs the packages Colab does not ship with.
import os, sys, subprocess, warnings
warnings.filterwarnings("ignore")
REPO = "https://github.com/GnsShubham06/resume-analyzer-nlp"
if "google.colab" in sys.modules:
    if not os.path.isdir("/content/resume-analyzer-nlp"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, "/content/resume-analyzer-nlp"], check=True)
    os.chdir("/content/resume-analyzer-nlp")
    extra = ['gensim']
    if extra:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *extra], check=True)
else:  # local run from notebooks/experiments/
    for _ in range(3):
        if os.path.isfile("requirements.txt"):
            break
        os.chdir("..")
sys.path.insert(0, os.getcwd())

import nltk
for pkg in ['punkt', 'punkt_tab']:
    nltk.download(pkg, quiet=True)

%config InlineBackend.figure_format = "retina"
import pandas as pd
pd.set_option("display.max_colwidth", 120)
df = pd.read_csv("data/resume_dataset.csv")
df["extracted_text"] = df["extracted_text"].fillna("")
print("Project folder:", os.path.basename(os.getcwd()), "| resumes:", len(df))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 9, "axes.titlesize": 10})
ORDER = ["Engineering", "HR", "Finance", "Healthcare", "Agriculture"]
SAMPLE_IDS = {"Engineering": "ENG_06", "HR": "HR_06", "Finance": "FIN_07",
              "Healthcare": "HLT_04", "Agriculture": "AGR_02"}     # five fictional resumes
splits = pd.read_csv("data/processed/splits.csv")[["resume_id", "split"]]
df = df.merge(splits, on="resume_id")

print("resumes per split:", df["split"].value_counts().to_dict())
print("classification_text (cleaned, lemmatised, stop words removed) of FIN_07:")
print(df.loc[df.resume_id == "FIN_07", "classification_text"].iloc[0][:260], "...")

## a. N-gram model
Counts of uni-, bi- and tri-grams, then one **bigram language model per domain** (Laplace smoothing) trained on the 60 training resumes. A new sentence gets the lowest perplexity under the model of its own domain.

In [ ]:
from collections import Counter
from nltk.util import ngrams
def tokens(text):
    return [w.lower() for w in nltk.word_tokenize(text) if any(c.isalnum() for c in w)]
def sentences(text):
    return [tokens(l) for l in str(text).split("\n") if len(l.split()) >= 3]
rows = []
for n in (1, 2, 3):   # counted on the cleaned, lemmatised text without stop words
    c = Counter(g for t in df.classification_text for g in ngrams(t.split(), n))
    rows.append({"n": n, "distinct n-grams": len(c),
                 "most frequent (count)": ", ".join(f"{' '.join(g)} ({k})" for g, k in c.most_common(4))})
display(pd.DataFrame(rows).set_index("n"))

In [ ]:
from nltk.lm import Laplace, Vocabulary
from nltk.lm.preprocessing import padded_everygram_pipeline, pad_both_ends
train = df[df.split == "train"]
all_sents = [s for t in train.cleaned_text for s in sentences(t)]
# one shared vocabulary, so that add-one smoothing treats every domain model the same way
vocab = Vocabulary([w for s in all_sents for w in s] + ["<s>", "</s>"], unk_cutoff=1)
lms = {}
for cat in ORDER:
    sents = [s for t in train[train.category == cat].cleaned_text for s in sentences(t)]
    data, _ = padded_everygram_pipeline(2, sents)
    lms[cat] = Laplace(2, vocabulary=vocab)
    lms[cat].fit(data)
def perplexity(lm, sentence):
    padded = list(pad_both_ends(tokens(sentence), n=2))
    return lm.perplexity(list(ngrams(padded, 2)))
new = {  # new sentences written by the team (not in the dataset)
    "Engineering": "Designed a steel bracket in SolidWorks and checked the welds before testing.",
    "HR": "Conducted exit interviews and updated the employee handbook for new joiners.",
    "Finance": "Reviewed vendor invoices and prepared the quarterly tax provision.",
    "Healthcare": "Monitored patient vitals in the ward and recorded medication doses.",
    "Agriculture": "Advised farmers on soil testing and drip irrigation for the rabi crop.",
}
ppl = pd.DataFrame({cat: [perplexity(lms[cat], s) for s in new.values()] for cat in ORDER},
                   index=[f"{d}: {s[:34]}..." for d, s in new.items()]).round(0).astype(int)
ppl["lowest perplexity"] = ppl[ORDER].idxmin(axis=1)
ppl.index.name = "sentence (true domain)"
display(ppl)
fin = lms["Finance"]
follow = fin.counts[["variance"]]
print("Finance bigram model  P(w | 'variance'):",
      [(w, round(fin.score(w, ["variance"]), 3)) for w, _ in follow.most_common(4)])
print("correct domain for", sum(ppl["lowest perplexity"] == pd.Series(list(new), index=ppl.index)), "of 5 new sentences")

## b. TF-IDF model

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, max_df=0.9)
X_tfidf = vec.fit_transform(df.classification_text)
vocab = vec.get_feature_names_out()
rows = []
for dom, rid in SAMPLE_IDS.items():
    i = df.index[df.resume_id == rid][0]
    w = X_tfidf[i].toarray().ravel()
    rows.append((rid, dom, ", ".join(f"{vocab[j]} ({w[j]:.2f})" for j in w.argsort()[::-1][:5])))
display(pd.DataFrame(rows, columns=["resume", "domain", "top TF-IDF terms (weight)"]))
print(f"TF-IDF matrix: {X_tfidf.shape[0]} resumes x {X_tfidf.shape[1]:,} uni/bi-gram features "
      f"({X_tfidf.nnz / X_tfidf.shape[0]:.0f} non-zero per resume)")

In [ ]:
S = cosine_similarity(X_tfidf)
np.fill_diagonal(S, np.nan)                      # ignore a resume's similarity with itself
cats = df.category.to_numpy()
M = pd.DataFrame([[np.nanmean(S[np.ix_(cats == a, cats == b)]) for b in ORDER] for a in ORDER],
                 index=ORDER, columns=ORDER)
fig, ax = plt.subplots(figsize=(4.8, 3.6))
im = ax.imshow(M.values, cmap="Greys", vmin=0, vmax=M.values.max())
for i in range(5):
    for j in range(5):
        ax.text(j, i, f"{M.values[i, j]:.2f}", ha="center", va="center",
                color="white" if M.values[i, j] > 0.6 * M.values.max() else "black", fontsize=8)
ax.set_xticks(range(5), ORDER, rotation=30, ha="right"); ax.set_yticks(range(5), ORDER)
ax.set_title("Mean TF-IDF cosine similarity between domains")
fig.colorbar(im, ax=ax, shrink=0.8); plt.tight_layout(); plt.show()

## c. Word embeddings – Word2Vec (trained on our resumes) and GloVe (pretrained)

In [ ]:
from gensim.models import Word2Vec
import gensim.downloader as api
sents = [t.split() for t in df.classification_text]
w2v = Word2Vec(sents, vector_size=100, window=5, min_count=2, sg=1, epochs=60, seed=42, workers=1)
glove = api.load("glove-wiki-gigaword-50")            # 400k words, 50 dimensions (~66 MB download)
probes = ["payroll", "nurse", "audit", "irrigation", "autocad"]
near = lambda kv, w: ", ".join(s for s, _ in kv.most_similar(w, topn=4)) if w in kv else "(not in vocabulary)"
rows = [(w, near(w2v.wv, w), near(glove, w)) for w in probes]
display(pd.DataFrame(rows, columns=["word", "Word2Vec: nearest words (our 100 resumes)",
                                    "GloVe: nearest words (Wikipedia + news)"]))
print(f"Word2Vec: {len(w2v.wv):,} words x {w2v.vector_size} dims (skip-gram, window 5) | "
      f"GloVe: {len(glove):,} words x {glove.vector_size} dims")

## d. Contextual embeddings – BERT
A static embedding gives a word the same vector in every sentence; BERT gives a different vector for each context.

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()
btok = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
bert = AutoModel.from_pretrained("google-bert/bert-base-uncased").eval()
def word_vector(sentence, word):
    enc = btok(sentence, return_tensors="pt")
    with torch.no_grad():
        h = bert(**enc).last_hidden_state[0]
    ids, target = enc["input_ids"][0].tolist(), btok(word, add_special_tokens=False)["input_ids"]
    i = next(k for k in range(len(ids)) if ids[k:k + len(target)] == target)
    return h[i:i + len(target)].mean(0)
pairs = [
    ("care", "Provided wound care to patients in the surgical ward.",
             "Delivered post-operative care in the cardiac unit.", "yes"),
    ("care", "Provided wound care to patients in the surgical ward.",
             "Handled customer care calls for retail banking clients.", "no"),
    ("plant", "Supervised maintenance of the bottling plant conveyor line.",
              "Advised farmers on plant protection against stem borer.", "no"),
    ("field", "Conducted field trials of drought-tolerant wheat varieties.",
              "Filled in every mandatory field of the payroll form.", "no"),
]
cos = torch.nn.functional.cosine_similarity
rows = [(w, a, b, same, round(float(cos(word_vector(a, w), word_vector(b, w), dim=0)), 3), 1.0)
        for w, a, b, same in pairs]
display(pd.DataFrame(rows, columns=["word", "sentence A", "sentence B", "same sense?", "BERT cosine",
                                    "Word2Vec / GloVe cosine"]))

## e. Which representation works best as features?
Same classifier for all (logistic regression), same stratified 5-fold cross-validation over the 100 resumes. TF-IDF and n-gram vectorizers are fitted inside each fold.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
y = df.category.to_numpy(dtype=object)
def mean_vec(kv, toks, dim):
    v = [kv[w] for t in toks for w in t.split("_") if w in kv]
    return np.mean(v, axis=0) if v else np.zeros(dim)
X_w2v = np.vstack([mean_vec(w2v.wv, t, 100) for t in sents])
X_glove = np.vstack([mean_vec(glove, t, 50) for t in sents])
def doc_embedding(text):                          # mean-pooled BERT over the first 512 tokens
    enc = btok(text, truncation=True, max_length=512, return_tensors="pt")
    with torch.no_grad():
        return bert(**enc).last_hidden_state[0].mean(0).numpy()
X_bert = np.vstack([doc_embedding(t) for t in df.transformer_text])
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
dense = lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))
reps = [("Bag of n-grams (1-2, counts)", df.classification_text,
         make_pipeline(CountVectorizer(ngram_range=(1, 2)), LogisticRegression(max_iter=3000, C=10))),
        ("TF-IDF (1-2 grams)", df.classification_text,
         make_pipeline(TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, max_df=0.9),
                       LogisticRegression(max_iter=3000, C=10))),
        ("Word2Vec 100-d (mean)", X_w2v, dense()), ("GloVe 50-d (mean)", X_glove, dense()),
        ("BERT-base 768-d (mean-pooled)", X_bert, dense())]
rows = []
for name, X, clf in reps:
    pred = cross_val_predict(clf, X, y, cv=skf)
    rows.append((name, accuracy_score(y, pred), f1_score(y, pred, average="macro")))
res = pd.DataFrame(rows, columns=["representation", "accuracy", "macro-F1"]).round(3)
display(res)
fig, ax = plt.subplots(figsize=(6.2, 2.6))
yy = np.arange(len(res))
ax.barh(yy - 0.18, res["accuracy"], 0.36, color="#444444", label="accuracy")
ax.barh(yy + 0.18, res["macro-F1"], 0.36, color="white", edgecolor="black", hatch="///", label="macro-F1")
ax.set_yticks(yy, res["representation"]); ax.invert_yaxis(); ax.set_xlim(0.5, 1.0)
ax.set_title("Same logistic regression, 5-fold CV, 100 resumes")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.13), ncol=2, fontsize=7, frameon=False)
plt.tight_layout(); plt.show()